# 🚀 LOGI-LLM: Adaptador Gemma-4-E4B para Google Colab com Interface Gradio

Este notebook foi desenvolvido para rodar o modelo **Gemma-4-E4B-Uncensored** com aceleração por GPU (T4/A100) gratuita no Google Colab, provendo uma interface de chat premium em tempo real usando o **Gradio**.

### ⚠️ **PASSO IMPORTANTE ANTES DE COMEÇAR**:
Certifique-se de que a sua máquina do Colab está com suporte a GPU ativo:
1. No menu superior, clique em **Ambiente de execução** (Runtime) > **Alterar tipo de ambiente de execução** (Change runtime type).
2. Em **Acelerador de hardware** (Hardware accelerator), selecione **GPU T4** (ou superior).
3. Clique em **Salvar**.

## 🛠️ Passo 1: Instalação das Dependências

Para extrair a máxima velocidade da placa de vídeo gratuita do Google Colab, vamos instalar o `llama-cpp-python` com suporte nativo à placa de vídeo (CUDA).

Oferecemos duas opções abaixo:
- **Opção 1 (Recomendada)**: Instala instantaneamente (15 segundos) usando pacotes pré-compilados.
- **Opção 2**: Compila direto do código-fonte caso a primeira dê algum erro de compatibilidade.

In [ ]:
# OPÇÃO 1: RÁPIDA (Instala em 15 segundos usando a versão pré-compilada para CUDA 12.1/12.2)
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

# OPÇÃO 2: UNIVERSAL (Demora cerca de 3 a 5 minutos pois compila tudo do zero)
# !CMAKE_ARGS="-DLLAMA_CUDA=on" FORCE_CMAKE=1 pip install llama-cpp-python --no-cache-dir

# Instalar Gradio para a interface web e Hugging Face Hub
!pip install gradio huggingface-hub

## ⚡ Passo 2: Inicializar o Modelo Gemma e Lançar a Interface

Agora, vamos carregar o modelo **Gemma-4-E4B-Uncensored** (descarregando 100% dos cálculos na GPU com `n_gpu_layers=-1`) e lançar a nossa interface Gradio.

Ao rodar o código abaixo, o Gradio vai gerar:
- Uma interface integrada diretamente aqui na célula do Colab.
- Um **link público temporário** (`xxxx.gradio.live`) que você pode abrir em qualquer aba do navegador ou até no celular!

In [ ]:
from llama_cpp import Llama
import gradio as gr

# 1. Inicializa o modelo com suporte a GPU
print("📥 Baixando e carregando o modelo na GPU...")
llm = Llama.from_pretrained(
    repo_id="HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive",
    filename="Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-IQ3_M.gguf",
    n_ctx=2048,       # Janela de contexto de 2048 tokens
    n_gpu_layers=-1   # Descarrega 100% das camadas na GPU T4
)
print("✅ Modelo pronto na placa de vídeo!")

# 2. Função de chat que suporta streaming em tempo real
def chat_respond(message, history, system_prompt, temperature, max_tokens, top_p, repeat_penalty):
    messages = []
    
    # Adiciona system prompt para ajustar a personalidade do chat
    if system_prompt.strip():
        messages.append({"role": "system", "content": system_prompt})
    
    # Adiciona o histórico existente
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    
    # Adiciona a nova mensagem
    messages.append({"role": "user", "content": message})
    
    try:
        response_stream = llm.create_chat_completion(
            messages=messages,
            temperature=float(temperature),
            max_tokens=int(max_tokens),
            top_p=float(top_p),
            repeat_penalty=float(repeat_penalty),
            stream=True
        )
        
        partial_text = ""
        for chunk in response_stream:
            delta = chunk['choices'][0]['delta']
            if 'content' in delta:
                partial_text += delta['content']
                yield partial_text
    except Exception as e:
        yield f"⚠️ Ocorreu um erro ao gerar a resposta: {str(e)}"

# 3. Criando a estrutura visual com o tema Soft
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="indigo")) as demo:
    with gr.Row():
        gr.Markdown(
            """
            # 🚀 LOGI-LLM: Gemma-4-E4B Uncensored
            ### Interface Inteligente rodando com Aceleração de GPU T4 no Google Colab!
            """
        )
        
    with gr.Row():
        # Sidebar lateral para parâmetros
        with gr.Column(scale=3, variant="panel"):
            gr.Markdown("### ⚙️ Parâmetros da IA")
            system_prompt = gr.Textbox(
                value="Você é um assistente de IA prestativo, inteligente e responde de forma clara em Português.",
                label="System Prompt (Personalidade)",
                lines=4
            )
            with gr.Accordion("⚙️ Ajustes Avançados", open=True):
                temperature = gr.Slider(minimum=0.1, maximum=1.5, value=0.7, step=0.05, label="Temperatura (Criatividade)")
                max_tokens = gr.Slider(minimum=64, maximum=2048, value=512, step=64, label="Max Tokens (Tamanho da resposta)")
                top_p = gr.Slider(minimum=0.1, maximum=1.0, value=0.9, step=0.05, label="Top P")
                repeat_penalty = gr.Slider(minimum=1.0, maximum=1.5, value=1.1, step=0.05, label="Penalidade de Repetição")
            
            gr.HTML("<hr style='border: 0; border-top: 1px solid rgba(255,255,255,0.1); margin: 15px 0;'>")
            gr.Markdown("⚡ **Aceleração**: CUDA GPU active\n💾 **Modelo**: Gemma-4-E4B")
            
        # Janela do Chatbot principal
        with gr.Column(scale=7):
            chatbot = gr.ChatInterface(
                fn=chat_respond,
                additional_inputs=[system_prompt, temperature, max_tokens, top_p, repeat_penalty],
                chatbot=gr.Chatbot(height=520, placeholder="🤖 **Como posso ajudar você hoje? Digite qualquer pergunta!**"),
                textbox=gr.Textbox(placeholder="Digite sua mensagem aqui...", container=False, scale=7),
                submit_btn="Enviar 🚀",
                retry_btn="Tentar Novamente 🔄",
                clear_btn="Limpar Chat 🗑️",
            )

# 4. Inicia a fila e lança o túnel público do Gradio
demo.queue().launch(share=True, debug=True)